# TabDPT Regressor — DIMER E2E tutorial

[GitHub](https://github.com/kurtvalcorza/tabdpt-regressor-pipeline) · [Open in Colab](https://colab.research.google.com/github/kurtvalcorza/tabdpt-regressor-pipeline/blob/main/tutorials/tabdpt_regressor_colab.ipynb) · [Model](https://huggingface.co/Layer6/TabDPT) · [Upstream](https://github.com/layer6ai-labs/TabDPT-inference)

**Profile:** `E2E` · **DIMER Notebook Specification:** 1.0

This notebook runs supervised tabular **regression** through the repository API. TabDPT is an in-context foundation model: `fit()` fits preprocessing and registers labelled support context; it does **not** gradient-train or fine-tune the pretrained weights. The upstream project supplies TabDPT and its weights; this repository adds immutable provenance, SHA-256 verification, schema-safe preprocessing, deterministic controls, DIMER artifact integration, and regression evaluation.

By the end, you will verify the exact model, load a public sample or gated BYOD CSV, validate and split data, inspect missing-value and capacity behavior, compare a training-mean baseline, evaluate MAE/RMSE/R², score new rows, export CSV/JSON results, export the DIMER serving artifact, and reload it from serialized state.

**Boundaries.** Regression only; no classification, gradient fine-tuning, or calibrated per-prediction uncertainty intervals. Sample metrics are tutorial sanity evidence, not benchmark or production evidence; upstream pretraining overlap with the public sample cannot be ruled out.

**Prerequisites.** Use a clean Google Colab runtime with **Python 3.11+** that can resolve the exact tutorial lock set. GPU is recommended; CPU is supported but slower. Internet access is required for repository/model acquisition unless cached. FlashAttention is disabled for Tesla T4 portability. BYOD remains in the runtime; do not upload restricted data to an unauthorized environment.


In [ ]:
from pathlib import Path
import shutil, sys
if sys.version_info < (3, 11):
    raise RuntimeError("This tutorial lock set requires Python 3.11+ (NumPy 2.3.0 upstream reproduction pin).")
REPO_DIR = Path("/content/tabdpt-regressor-pipeline")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/kurtvalcorza/tabdpt-regressor-pipeline.git /content/tabdpt-regressor-pipeline
%pip install -q -r /content/tabdpt-regressor-pipeline/tutorials/requirements-colab.txt
%pip install -q --no-deps /content/tabdpt-regressor-pipeline
!git -C /content/tabdpt-regressor-pipeline rev-parse HEAD


## 1. Runtime and immutable model provenance

The lock set is installed before model imports. The next cell prints the effective runtime, resolves the immutable Hugging Face revision, and verifies the model weight SHA-256. A matching digest establishes byte integrity against this repository's expected model artifact; it does not establish model quality or producer authenticity.


In [ ]:
import csv, io, json, platform
import importlib.metadata as md
import numpy as np, pandas as pd, torch
from tabdpt_regressor_pipeline import (
    TABDPT_HF_REPO, TABDPT_HF_REVISION, TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME, TABDPT_WEIGHT_SHA256, TabDPTRegressionPipeline,
    export_artifact_bundle, load_verified_artifact, resolve_tabdpt_weights,
)
print("Python", platform.python_version(), "torch", md.version("torch"), "tabdpt", md.version("tabdpt"),
      "numpy", md.version("numpy"), "pandas", md.version("pandas"), "sklearn", md.version("scikit-learn"))
print("Device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "use_flash=False")
weights = resolve_tabdpt_weights()
print({"repo": TABDPT_HF_REPO, "revision": TABDPT_HF_REVISION, "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT,
       "filename": TABDPT_WEIGHT_FILENAME, "sha256": TABDPT_WEIGHT_SHA256, "verifiedPath": str(weights)})


## 2. Sample/BYOD, validation, preprocessing visibility, and leakage-aware split

Default data is scikit-learn's public diabetes regression sample. For BYOD, the expected contract is one CSV with unique feature names and a finite numeric column named `target`. Set `USE_BYOD=True`. If identifier-like categorical fields contain numeric-looking strings such as `01`, list those fields in `CATEGORICAL_COLUMNS` **before upload** so CSV parsing preserves their string identity; every named categorical column must exist in the header. All other CSV columns use normal pandas inference.

Repository preprocessing preserves numeric columns and deterministically maps categorical values. Categorical missing values receive a fitted missing code and unseen categories receive a separate unknown code. Numeric missing values remain NaN through the repository encoder and are mean-imputed by TabDPT's upstream fitted imputer; upstream standardization is also fitted on support/training rows. Infinite numeric feature values fail before conditioning. These fitted states are serialized for verified artifact reload.

Random 80/20 splitting assumes rows are sufficiently independent; use preserved temporal, group, or spatial boundaries for leakage-sensitive data. No model selection uses this holdout. `context_size` limits support rows used per prediction. If encoded feature width exceeds the loaded model's `max_features`, upstream feature reduction is surfaced after conditioning rather than silently truncating features.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
USE_BYOD = False  # @param {type:"boolean"}
CATEGORICAL_COLUMNS = []  # @param {type:"raw"}
def read_csv_checked(raw, categorical_columns):
    text = raw.decode("utf-8-sig")
    header = next(csv.reader(io.StringIO(text)), [])
    duplicates = sorted({x for x in header if header.count(x) > 1})
    if duplicates:
        raise ValueError(f"Duplicate CSV columns: {duplicates}")
    missing_declared = [c for c in categorical_columns if c not in header]
    if missing_declared:
        raise ValueError(f"CATEGORICAL_COLUMNS not present in CSV header: {missing_declared}")
    return pd.read_csv(io.BytesIO(raw), dtype={c: "string" for c in categorical_columns})
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one CSV.")
    name, raw = next(iter(uploaded.items()))
    if not name.lower().endswith(".csv"):
        raise ValueError("BYOD must be CSV.")
    frame = read_csv_checked(raw, CATEGORICAL_COLUMNS)
    data_source = f"user CSV: {name}"
else:
    frame = load_diabetes(as_frame=True).frame
    data_source = "scikit-learn diabetes sample"
TARGET, SEED, CONTEXT_SIZE, N_ENSEMBLES, BATCH_SIZE = "target", 42, 512, 2, 512
if frame.columns.duplicated().any() or TARGET not in frame:
    raise ValueError("Unique columns and the target column are required.")
y = pd.to_numeric(frame[TARGET], errors="coerce")
if y.isna().any() or not np.isfinite(y.to_numpy(float)).all() or y.nunique() < 2:
    raise ValueError("Target must be finite, numeric, and non-constant.")
frame[TARGET] = y
features = frame.drop(columns=[TARGET])
for column in features.select_dtypes(include=np.number).columns:
    values = features[column].dropna().to_numpy(dtype=float)
    if values.size and not np.isfinite(values).all():
        raise ValueError(f"Numeric feature {column!r} contains infinite values.")
missing_counts = features.isna().sum()
print("Rows/features:", len(frame), features.shape[1])
print("Columns with missing values:", missing_counts[missing_counts > 0].to_dict() or "none")
print("Input dtypes:", features.dtypes.astype(str).to_dict())
print("Declared categorical columns:", CATEGORICAL_COLUMNS if USE_BYOD else "sample schema")
train, test = train_test_split(frame, test_size=0.2, random_state=SEED)
print(data_source, "train", train.shape, "holdout", test.shape)
print("Requested context_size:", CONTEXT_SIZE, "effective support rows <=", min(len(train), CONTEXT_SIZE))
if len(train) > CONTEXT_SIZE:
    print("Support exceeds context_size; seeded upstream context subsampling applies.")


## 3. Baseline, in-context conditioning, capacity report, and evaluation

The training-mean constant predictor is the trivial baseline. MAE is average absolute error in target units; RMSE is also in target units and weights large errors more; R² is relative to a constant-mean reference and can be negative. These are single-holdout tutorial estimates with no dispersion claim.

`fit()` performs preprocessing fitting plus in-context conditioning, **not gradient training**. After conditioning, the notebook prints encoded feature count, model feature ceiling, active feature-reduction method, and missing-value preprocessing. Seeds control supported stochastic paths; bitwise determinism across hardware kernels is not promised.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y_true = test[TARGET].to_numpy(float)
baseline_pred = np.full(len(test), train[TARGET].mean())
baseline = {"mae": float(mean_absolute_error(y_true, baseline_pred)),
            "rmse": float(np.sqrt(mean_squared_error(y_true, baseline_pred))),
            "r2": float(r2_score(y_true, baseline_pred))}
pipe = TabDPTRegressionPipeline(model_weight_path=weights, compile_model=False, use_flash=False, seed=SEED)
pipe.fit(train, target_column=TARGET, seed=SEED)
encoded_feature_count = len(pipe.feature_encoder.feature_columns)
model_feature_ceiling = int(pipe.estimator.max_features)
feature_reduction = str(pipe.estimator.feature_reduction)
numeric_missing_columns = [c for c in pipe.feature_encoder.feature_columns
                           if c in pipe.feature_encoder.numeric_columns and train[c].isna().any()]
print({"encodedFeatureCount": encoded_feature_count, "modelFeatureCeiling": model_feature_ceiling,
       "featureReduction": feature_reduction, "featureReductionActive": encoded_feature_count > model_feature_ceiling,
       "numericMissingColumns": numeric_missing_columns, "upstreamImputer": type(pipe.estimator.imputer).__name__})
if encoded_feature_count > model_feature_ceiling:
    print(f"Feature count exceeds {model_feature_ceiling}; upstream {feature_reduction} reduction is active.")
else:
    print("Feature reduction is not active for this dataset.")
if numeric_missing_columns:
    print("Numeric missing values are mean-imputed using support/training-fitted statistics.")
kw = {"n_ensembles": N_ENSEMBLES, "context_size": CONTEXT_SIZE, "batch_size": BATCH_SIZE, "seed": SEED}
metrics = pipe.evaluate(test, **kw)
print("tutorial TabDPT", metrics)
print("training-mean baseline", baseline)


## 4. New-data inference and machine-readable outputs

The target is removed before scoring. Training-fitted repository encoding and upstream imputation/scaling are reused; no preprocessing is fitted on the new rows. Predictions are continuous point estimates only. `row_id` remains outside the model feature schema and maps exported predictions back to inputs. Provenance records immutable model identity, runtime, split, inference controls, capacity, and preprocessing semantics.


In [ ]:
new_rows = test.drop(columns=[TARGET]).head(8).copy()
pred = pipe.predict(new_rows, **kw)
out = pd.DataFrame({"row_id": new_rows.index.to_numpy(), "prediction": pred.to_numpy()})
OUT = Path("/content/tabdpt-tutorial-output")
OUT.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT / "tabdpt_regression_predictions.csv", index=False)
(OUT / "tabdpt_regression_metrics.json").write_text(json.dumps({"tabdpt": metrics, "training_mean_baseline": baseline}, indent=2) + "\n")
provenance = {
    "model": {"repo": TABDPT_HF_REPO, "revision": TABDPT_HF_REVISION, "filename": TABDPT_WEIGHT_FILENAME,
              "sha256": TABDPT_WEIGHT_SHA256, "upstreamCodeCommit": TABDPT_UPSTREAM_CODE_COMMIT},
    "runtime": {"python": platform.python_version(), "torch": md.version("torch"), "tabdpt": md.version("tabdpt"),
                "numpy": md.version("numpy"), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
                "use_flash": False},
    "data": {"source": data_source, "split": "single seeded 80/20 random holdout", "seed": SEED,
             "trainRows": len(train), "holdoutRows": len(test)},
    "preprocessing": {"encodedFeatureCount": encoded_feature_count, "modelFeatureCeiling": model_feature_ceiling,
                      "featureReduction": feature_reduction, "featureReductionActive": encoded_feature_count > model_feature_ceiling,
                      "numericMissingPolicy": "support/training-fitted mean imputation",
                      "categoricalMissingPolicy": "dedicated fitted missing code",
                      "unknownCategoryPolicy": "dedicated fitted unknown code",
                      "declaredCategoricalColumns": CATEGORICAL_COLUMNS if USE_BYOD else []},
    "inference": kw,
}
(OUT / "tabdpt_regression_provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
print(out.head())
print("Wrote prediction CSV plus metrics/provenance JSON.")


## 5. Export and fresh-boundary verification

For this in-context model the deployable serving state is not the checkpoint alone: it includes labelled support data, fitted repository/upstream preprocessing state, and the exact pinned base-model contract. `training_context.parquet` inherits the source data's confidentiality, licensing, retention, and disclosure obligations.

The artifact is copied to a fresh directory, validated before reconstruction, and loaded through the verified **no-preprocessing-refit** path. Predictions must match within `rtol=1e-5`, `atol=1e-6`; bitwise identity is not required.


In [ ]:
ART = OUT / "artifact"
manifest_path = export_artifact_bundle(pipe, train, ART)
RELOAD = Path("/content/tabdpt-artifact-reload")
if RELOAD.exists():
    shutil.rmtree(RELOAD)
shutil.copytree(ART, RELOAD)
reloaded = load_verified_artifact(RELOAD / "artifact.json", model_weight_path=weights, compile_model=False, use_flash=False, seed=SEED)
assert reloaded.preprocessing_restored_ is True, "Verified reload must restore fitted preprocessing state without refit."
pred2 = reloaded.predict(new_rows, **kw)
np.testing.assert_allclose(pred.to_numpy(), pred2.to_numpy(), rtol=1e-5, atol=1e-6)
print("PASS: fitted preprocessing restored without refit and predictions are equivalent (rtol=1e-5, atol=1e-6).")


## Interpretation, limits, and next steps

A successful run proves this repository can resolve and digest-check its pinned TabDPT weight, validate and condition on regression support data, surface missing-value/capacity behavior, compute sample metrics against a trivial baseline, score new rows, export machine-readable results and DIMER serving state, and reconstruct equivalent predictions from serialized fitted preprocessing without refitting it.

It does **not** establish benchmark superiority, domain generalization, fairness, robustness, calibration, production safety, or deployment fitness. For real data, preserve domain-appropriate splits, compare task-relevant baselines, test representative inputs and missing-data patterns, inspect whether feature/context reduction activates, and complete the repository's recorded clean-runtime/on-platform release verification.
